In [1]:
!pip install transformers torch

In [2]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "gpt2-medium"

print("Loading model, please wait...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
tokenizer.pad_token = tokenizer.eos_token

# ── Knowledge base for common factual questions ──────────────
KNOWLEDGE_BASE = {
    "who created python": "Python was created by Guido van Rossum and first released in 1991.",
    "who invented python": "Python was created by Guido van Rossum and first released in 1991.",
    "what is python": "Python is a high-level, interpreted programming language known for its simplicity and readability.",
    "what is artificial intelligence": "Artificial Intelligence (AI) is the simulation of human intelligence by machines, enabling them to learn, reason, and solve problems.",
    "what is machine learning": "Machine Learning is a subset of AI where systems learn from data to improve their performance without being explicitly programmed.",
    "what is deep learning": "Deep Learning is a subset of machine learning that uses neural networks with many layers to model complex patterns in data.",
    "who created java": "Java was created by James Gosling at Sun Microsystems and released in 1995.",
    "who invented java": "Java was created by James Gosling at Sun Microsystems and released in 1995.",
    "what is java": "Java is a high-level, object-oriented programming language designed to be platform-independent.",
    "who created c++": "C++ was created by Bjarne Stroustrup in 1979 as an extension of the C language.",
    "what is c++": "C++ is a general-purpose programming language with object-oriented features, widely used in systems and game development.",
    "who invented the internet": "The Internet was developed by ARPA in the 1960s. Tim Berners-Lee invented the World Wide Web in 1989.",
    "what is the internet": "The Internet is a global network of computers that communicate using standardized protocols to share information.",
    "who is the father of computer science": "Alan Turing is widely regarded as the father of computer science and artificial intelligence.",
    "what is a neural network": "A neural network is a computational model inspired by the human brain, made up of layers of interconnected nodes used in AI.",
    "what is nlp": "Natural Language Processing (NLP) is a branch of AI that enables computers to understand, interpret, and generate human language.",
    "hello": "Hello! Nice to meet you. How can I assist you today?",
    "hi": "Hi there! How can I help you?",
    "how are you": "I'm doing great, thank you for asking! How can I help you today?",
    "thank you": "You're welcome! Feel free to ask more questions.",
    "thanks": "You're welcome! Is there anything else I can help you with?",
    "bye": "Goodbye! Have a great day!",
    "good morning": "Good morning! How can I assist you today?",
    "good night": "Good night! Take care!",
}


def get_kb_response(user_text):
    """Check knowledge base for a matching answer."""
    cleaned = user_text.lower().strip().rstrip("?!")
    for key, answer in KNOWLEDGE_BASE.items():
        if key in cleaned:
            return answer
    return None


def generate_response(user_text, chat_history: list):
    """Fall back to GPT-2 for questions not in the knowledge base."""
    prompt = ""
    for turn in chat_history[-4:]:
        prompt += f"Q: {turn['user']}\nA: {turn['bot']}\n\n"
    prompt += f"Q: {user_text}\nA:"

    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.encode("\n")[0],
        )

    new_tokens = output[0][inputs.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response = response.split("\n")[0].strip()
    return response if response else "I'm not sure about that. Could you rephrase?"


print("\n" + "=" * 50)
print("Chatbot: Hello! I am your AI assistant.")
print("How can I help you today?")
print("(type 'exit' or 'quit' to end)")
print("=" * 50 + "\n")

chat_history = []

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in {"exit", "quit"}:
        print("\nChatbot: It was great talking with you. Goodbye!\n")
        break

    # Check knowledge base first, then fall back to GPT-2
    response = get_kb_response(user_input) or generate_response(user_input, chat_history)
    chat_history.append({"user": user_input, "bot": response})

    print(f"\nChatbot: {response}\n")

Loading model, please wait...


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Chatbot: Hello! I am your AI assistant.
How can I help you today?
(type 'exit' or 'quit' to end)

You: Hello

Chatbot: Hello! Nice to meet you. How can I assist you today?

You: What is artificial intelligence?

Chatbot: Artificial Intelligence (AI) is the simulation of human intelligence by machines, enabling them to learn, reason, and solve problems.

You: Who created python?

Chatbot: Python was created by Guido van Rossum and first released in 1991.

You: Exit

Chatbot: It was great talking with you. Goodbye!

